In [2]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.cluster import DBSCAN
from sklearn.cluster import OPTICS
from hdbscan import HDBSCAN

from sklearn.metrics import silhouette_score
from hdbscan.validity import validity_index
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import adjusted_mutual_info_score

In [3]:
df = pd.read_csv(r"..//data/processed/Dataset.csv",index_col=0)
print(f'Data shape {df.shape}')
df.head()

Data shape (2087, 32)


,Gender_Female,Gender_Male,family_history_with_overweight_no,family_history_with_overweight_yes,FAVC_no,FAVC_yes,CAEC_Always,CAEC_Frequently,CAEC_Sometimes,CAEC_no,...,MTRANS_Walking,Age,Height,Weight,FCVC,NCP,CH2O,FAF,TUE,BMI
0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,-0.526613,-0.887408,-0.872985,-0.788364,0.390906,-0.007810,-1.186977,0.554211,-0.670475
1,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,-0.526613,-1.960788,-1.178508,1.082164,0.390906,1.636552,2.328908,-1.090505,-0.688960
2,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,-0.212507,1.044677,-0.376509,-0.788364,0.390906,-0.007810,1.156947,0.554211,-0.747890
3,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.415705,1.044677,0.005395,1.082164,0.390906,-0.007810,1.156947,-1.090505,-0.363194
4,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,-0.369560,0.830001,0.112328,-0.788364,-2.225418,-0.007810,-1.186977,-1.090505,-0.177412


In [37]:
label_true  = pd.read_csv(r"..//data/processed/label.csv",index_col=0).squeeze()
label_true.shape

(2087,)

In [5]:
train = np.load(r"../data/processed/df_success.npy")
train.shape

(2087, 32)

In [57]:
models = {
    'DBSCAN' : DBSCAN(
                    eps=2.1,
                    min_samples=20),
    
    'OPTICS' : OPTICS(min_samples=20),
    
    'HDBSCAN' : HDBSCAN(min_cluster_size=20,
                        min_samples=20)
}

In [58]:
result = []

for name, model in models.items():
    print(f"{name} model training")
    label = model.fit_predict(train)
    
    if len(np.unique(label) > 1):
        sil = silhouette_score(train,label)
    else:
        sil = np.nan
        
    result.append({
        'model' : name,
        "silhouette_score" : sil,
        'DBCV' : validity_index(train,label),
        'ARI' : adjusted_rand_score(label_true,label),
        "AMI" : adjusted_mutual_info_score(label_true,label)})
  
res = pd.DataFrame(result).set_index('model') 
res   

DBSCAN model training
OPTICS model training
HDBSCAN model training


,silhouette_score,DBCV,ARI,AMI
model,,,,
DBSCAN,0.031807,-0.156670,0.109845,0.232115
OPTICS,-0.104989,0.093482,0.035080,0.285749
HDBSCAN,-0.058057,0.125945,0.075340,0.349384
